# Notebook A v2 — Deterministic Trace Generator with Bit Manipulation

Purpose:
- Read competition `train.csv`.
- Generate SFT-ready deterministic reasoning traces.
- Preserve v1.1 categories: gravity, unit conversion, numeral.
- Add first bit-manipulation solver traces.
- Write `train_traces_v2_bit.jsonl`.

This notebook does **not** train and does **not** package a submission.

In [1]:
import json
import math
import re
import itertools
from pathlib import Path

import pandas as pd

TRACE_VERSION = "v2_bit"
WORKING_DIR = Path("/kaggle/working")
OUTPUT_JSONL = WORKING_DIR / f"train_traces_{TRACE_VERSION}.jsonl"
OUTPUT_SUMMARY_CSV = WORKING_DIR / f"trace_summary_{TRACE_VERSION}.csv"

TRAIN_CSV_CANDIDATES = [
    Path("/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"),
    Path("/kaggle/input/nvidia-nemotron-model-reasoning-challenge/train.csv"),
    Path("/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"),
]
if Path("/kaggle/input").exists():
    TRAIN_CSV_CANDIDATES.extend(Path("/kaggle/input").glob("**/train.csv"))

TRAIN_CSV = next((p for p in TRAIN_CSV_CANDIDATES if p.exists()), None)
if TRAIN_CSV is None:
    raise FileNotFoundError("Could not find train.csv. Add the competition dataset as an input.")

print("TRAIN_CSV:", TRAIN_CSV)
train = pd.read_csv(TRAIN_CSV)
print("train shape:", train.shape)
print(train.head())

TRAIN_CSV: /kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv
train shape: (9500, 3)
         id                                             prompt  \
0  00066667  In Alice's Wonderland, a secret bit manipulati...   
1  000b53cf  In Alice's Wonderland, a secret bit manipulati...   
2  00189f6a  In Alice's Wonderland, secret encryption rules...   
3  001b24c4  In Alice's Wonderland, numbers are secretly co...   
4  001c63cb  In Alice's Wonderland, secret encryption rules...   

                  answer  
0               10010111  
1               01000011  
2      cat imagines book  
3                XXXVIII  
4  wizard creates secret  


In [2]:
def classify_prompt(prompt: str) -> str:
    p = str(prompt)
    if "bit manipulation" in p or re.search(r"\b[01]{8}\s*->\s*[01]{8}\b", p):
        return "bit_manipulation"
    if "gravitational constant" in p or "falling distance" in p or "d = 0.5" in p:
        return "gravity"
    if "unit conversion" in p or ("becomes" in p and "measurement" in p):
        return "unit_conversion"
    if "numeral system" in p or "Wonderland numeral" in p or "Roman" in p:
        return "numeral"
    if "secret encryption rules are used on text" in p or "decrypt the following text" in p:
        return "cipher"
    if "secret set of transformation rules is applied to equations" in p:
        return "equation_symbolic"
    return "unknown"

train["category_detected"] = train["prompt"].map(classify_prompt)
print(train["category_detected"].value_counts())

category_detected
bit_manipulation     1602
gravity              1597
unit_conversion      1594
cipher               1576
numeral              1576
equation_symbolic    1555
Name: count, dtype: int64


In [3]:
# -----------------------------
# Category trace builders
# -----------------------------

def boxed(ans) -> str:
    return f"\\boxed{{{str(ans).strip()}}}"

def user_message(prompt: str) -> str:
    return str(prompt).strip() + "\nPlease put your final answer inside \\boxed{}. For example: \\boxed{your answer}"

def extract_gravity_values(prompt: str):
    examples = re.findall(r"For t = ([0-9.]+)s, distance = ([0-9.]+) m", prompt)
    target = re.search(r"falling distance for t = ([0-9.]+)s", prompt)
    return [(float(t), float(d)) for t, d in examples], float(target.group(1)) if target else None

def build_gravity_trace(row):
    examples, target_t = extract_gravity_values(row["prompt"])
    ans = str(row["answer"]).strip()
    if len(examples) < 2 or target_t is None:
        return None
    t1, d1 = examples[0]
    t2, d2 = examples[1]
    rate1 = d1 / (t1 * t1)
    rate2 = d2 / (t2 * t2)
    tgt_sq = target_t * target_t
    pred = rate1 * tgt_sq
    trace = (
        "Template: gravity. Ignore story text. Use RATE=d/t^2, where RATE=0.5g.\n"
        f"EX1: t={t1}, d={d1}. RATE={d1}/{t1}^2={rate1:.4f}.\n"
        f"Target: t={target_t}. t^2={tgt_sq:.4f}. RESULT={rate1:.4f}*{tgt_sq:.4f}={pred:.4f}.\n"
        f"VER: EX2 RATE={d2}/{t2}^2={rate2:.4f}; rates agree within tolerance.\n"
        f"Final answer: {boxed(ans)}"
    )
    return trace

def extract_unit_values(prompt: str):
    examples = re.findall(r"([0-9.]+) m becomes ([0-9.]+)", prompt)
    target = re.search(r"convert the following measurement:\s*([0-9.]+) m", prompt, flags=re.I)
    return [(float(x), float(y)) for x, y in examples], float(target.group(1)) if target else None

def build_unit_trace(row):
    examples, target_x = extract_unit_values(row["prompt"])
    ans = str(row["answer"]).strip()
    if len(examples) < 2 or target_x is None:
        return None
    x1, y1 = examples[0]
    x2, y2 = examples[1]
    rate1 = y1 / x1
    rate2 = y2 / x2
    pred = target_x * rate1
    trace = (
        "Template: unit conversion. Ignore story text. Use RATE=out/in.\n"
        f"EX1: in={x1}, out={y1}. RATE={y1}/{x1}={rate1:.4f}.\n"
        f"Target: in={target_x}. RESULT={target_x}*{rate1:.4f}={pred:.4f}.\n"
        f"VER: EX2 RATE={y2}/{x2}={rate2:.4f}; rates agree within tolerance.\n"
        f"Final answer: {boxed(ans)}"
    )
    return trace

def build_numeral_trace(row):
    ans = str(row["answer"]).strip()
    prompt = str(row["prompt"])
    target_int = re.search(r"(?:write the number|Convert)\s+([0-9]+)", prompt, flags=re.I)
    target_roman = re.search(r"Convert\s+([MDCLXVI]+)\s+to an integer", prompt)
    if target_int:
        target = target_int.group(1)
        mode = "integer-to-Roman"
    elif target_roman:
        target = target_roman.group(1)
        mode = "Roman-to-integer"
    else:
        target = "target"
        mode = "numeral"
    trace = (
        f"Template: numeral conversion ({mode}). Ignore story text.\n"
        f"Target: {target}. Apply standard Roman numeral conversion and verify by round trip.\n"
        f"Final answer: {boxed(ans)}"
    )
    return trace

In [4]:
# -----------------------------
# Bit manipulation solver v2
# -----------------------------

def parse_bit_problem(prompt: str):
    pairs = re.findall(r"\b([01]{8})\s*->\s*([01]{8})\b", str(prompt))
    tgt = re.search(r"determine the output for:\s*([01]{8})", str(prompt), flags=re.I)
    return pairs, tgt.group(1) if tgt else None

def bits(s: str):
    return [int(c) for c in s]

def eval_expr(expr, b):
    op = expr[0]
    if op == "C":
        return expr[1]
    if op == "ID":
        return b[expr[1]]
    if op == "NOT":
        return 1 - b[expr[1]]
    if op == "AND":
        return b[expr[1]] & b[expr[2]]
    if op == "OR":
        return b[expr[1]] | b[expr[2]]
    if op == "XOR":
        return b[expr[1]] ^ b[expr[2]]
    if op == "NAND":
        return 1 - (b[expr[1]] & b[expr[2]])
    if op == "NOR":
        return 1 - (b[expr[1]] | b[expr[2]])
    if op == "XNOR":
        return 1 - (b[expr[1]] ^ b[expr[2]])
    if op == "AND_NOT_A":
        return (1 - b[expr[1]]) & b[expr[2]]
    if op == "AND_NOT_B":
        return b[expr[1]] & (1 - b[expr[2]])
    if op == "OR_NOT_A":
        return (1 - b[expr[1]]) | b[expr[2]]
    if op == "OR_NOT_B":
        return b[expr[1]] | (1 - b[expr[2]])
    if op == "MAJ":
        return 1 if (b[expr[1]] + b[expr[2]] + b[expr[3]]) >= 2 else 0
    if op == "PAR3":
        return b[expr[1]] ^ b[expr[2]] ^ b[expr[3]]
    if op == "CHO":
        return b[expr[2]] if b[expr[1]] else b[expr[3]]
    if op == "AND_OR":
        return (b[expr[1]] & b[expr[2]]) | b[expr[3]]
    if op == "OR_AND":
        return (b[expr[1]] | b[expr[2]]) & b[expr[3]]
    if op == "AND_XOR":
        return (b[expr[1]] & b[expr[2]]) ^ b[expr[3]]
    if op == "OR_XOR":
        return (b[expr[1]] | b[expr[2]]) ^ b[expr[3]]
    if op == "XOR_AND":
        return (b[expr[1]] ^ b[expr[2]]) & b[expr[3]]
    if op == "XOR_OR":
        return (b[expr[1]] ^ b[expr[2]]) | b[expr[3]]
    raise ValueError(expr)

def expr_to_text(expr):
    op = expr[0]
    if op == "C":
        return f"C{expr[1]}"
    if op in {"ID", "NOT"}:
        return f"{op}({expr[1]})"
    return f"{op}({','.join(map(str, expr[1:]))})"

EXPR_LIBRARY = []
for c in [0, 1]:
    EXPR_LIBRARY.append(("C", c))
for i in range(8):
    EXPR_LIBRARY.extend([("ID", i), ("NOT", i)])
for i, j in itertools.permutations(range(8), 2):
    for op in ["AND", "OR", "XOR", "NAND", "NOR", "XNOR", "AND_NOT_A", "AND_NOT_B", "OR_NOT_A", "OR_NOT_B"]:
        EXPR_LIBRARY.append((op, i, j))
for i, j, k in itertools.permutations(range(8), 3):
    for op in ["MAJ", "PAR3", "CHO", "AND_OR", "OR_AND", "AND_XOR", "OR_XOR", "XOR_AND", "XOR_OR"]:
        EXPR_LIBRARY.append((op, i, j, k))

print("Bit expression library size:", len(EXPR_LIBRARY))

def solve_bit_problem(prompt: str):
    pairs, target = parse_bit_problem(prompt)
    if not pairs or target is None:
        return None

    in_bits = [bits(x) for x, _ in pairs]
    out_bits = [bits(y) for _, y in pairs]
    target_bits = bits(target)

    locks = []
    pred_bits = []

    for out_pos in range(8):
        target_col = [y[out_pos] for y in out_bits]
        found = None
        for expr in EXPR_LIBRARY:
            vals = [eval_expr(expr, x) for x in in_bits]
            if vals == target_col:
                found = expr
                break
        if found is None:
            return None

        tgt_val = eval_expr(found, target_bits)
        locks.append((out_pos, found, tgt_val))
        pred_bits.append(str(tgt_val))

    return "".join(pred_bits), locks, pairs, target

def build_bit_trace(row):
    solved = solve_bit_problem(row["prompt"])
    if solved is None:
        return None
    pred, locks, pairs, target = solved
    ans = str(row["answer"]).strip()

    # Only train on solver-validated bit rows. This prevents teaching wrong rules.
    if pred != ans:
        return None

    lock_lines = []
    for pos, expr, tgt_val in locks:
        lock_lines.append(f"B{pos}:{expr_to_text(expr)}->{tgt_val}")
    lock_text = " ".join(lock_lines)

    trace = (
        "Template: binary bit manipulation. Ignore story text. Solve each output bit independently.\n"
        f"Examples: {len(pairs)} pairs. Target input: {target}.\n"
        "For each bit, select the first expression that matches all example output bits, then apply it to target.\n"
        f"LOCKS: {lock_text}\n"
        f"Concatenate B0..B7 gives {ans}.\n"
        f"Final answer: {boxed(ans)}"
    )
    return trace

Bit expression library size: 3602


In [5]:
# -----------------------------
# Generate traces
# -----------------------------

TRACE_BUILDERS = {
    "gravity": build_gravity_trace,
    "unit_conversion": build_unit_trace,
    "numeral": build_numeral_trace,
    "bit_manipulation": build_bit_trace,
}

records = []
skipped = []

for row in train.to_dict("records"):
    category = row["category_detected"]
    builder = TRACE_BUILDERS.get(category)
    if builder is None:
        skipped.append({"id": row["id"], "category": category, "reason": "no_builder"})
        continue

    trace = builder(row)
    if trace is None:
        skipped.append({"id": row["id"], "category": category, "reason": "builder_returned_none"})
        continue

    answer = str(row["answer"]).strip()
    rec = {
        "id": row["id"],
        "category": category,
        "answer": answer,
        "trace_version": TRACE_VERSION,
        "approx_trace_chars": len(trace),
        "messages": [
            {"role": "user", "content": user_message(row["prompt"])},
            {"role": "assistant", "content": trace},
        ],
    }
    records.append(rec)

print("Generated records:", len(records))
summary = pd.DataFrame(records)["category"].value_counts().reset_index()
summary.columns = ["category", "count"]
print(summary)
print("Skipped:", len(skipped))
if skipped:
    print(pd.DataFrame(skipped)["category"].value_counts())

Generated records: 5635
           category  count
0           gravity   1597
1   unit_conversion   1594
2           numeral   1576
3  bit_manipulation    868
Skipped: 3865
category
cipher               1576
equation_symbolic    1555
bit_manipulation      734
Name: count, dtype: int64


In [6]:
# -----------------------------
# Validate and write outputs
# -----------------------------

def extract_boxed(text: str):
    matches = re.findall(r"\\boxed\{([^}]*)\}", str(text))
    return matches[-1].strip() if matches else None

bad = []
for rec in records:
    assistant = rec["messages"][1]["content"]
    pred = extract_boxed(assistant)

    if pred != str(rec["answer"]).strip():
        bad.append((rec["id"], rec["category"], rec["answer"], pred))

    # Detect actual Python backspace character, not the literal string "\b"
    if "\x08" in assistant:
        bad.append((rec["id"], rec["category"], rec["answer"], "contains_backspace"))

assert not bad[:5], f"Validation failures: {bad[:5]}"

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

summary = pd.DataFrame(records)[["id", "category", "answer", "approx_trace_chars"]]
summary.to_csv(OUTPUT_SUMMARY_CSV, index=False)

print("Wrote:", OUTPUT_JSONL)
print("Wrote:", OUTPUT_SUMMARY_CSV)
print("Final counts:")
print(summary["category"].value_counts())
print("Trace char stats:")
print(summary.groupby("category")["approx_trace_chars"].describe())

Wrote: /kaggle/working/train_traces_v2_bit.jsonl
Wrote: /kaggle/working/trace_summary_v2_bit.csv
Final counts:
category
gravity             1597
unit_conversion     1594
numeral             1576
bit_manipulation     868
Name: count, dtype: int64
Trace char stats:
                   count        mean        std    min    25%    50%    75%  \
category                                                                      
bit_manipulation   868.0  412.259217  13.931397  383.0  404.0  410.0  419.0   
gravity           1597.0  269.816531   2.640684  263.0  268.0  270.0  272.0   
numeral           1576.0  171.939086   1.636130  168.0  171.0  172.0  173.0   
unit_conversion   1594.0  250.216437   2.061565  241.0  249.0  251.0  252.0   

                    max  
category                 
bit_manipulation  456.0  
gravity           276.0  
numeral           176.0  
unit_conversion   252.0  


## Next notebook

Attach this Notebook A v2 output to Notebook B and set Notebook B to read:

`train_traces_v2_bit.jsonl`

Recommended first training run:
- keep LoRA rank 32
- keep LR `2e-4`
- start with `SMOKE_SAMPLE_SIZE = 256` or `512`
- output adapter name: `adapter_sft_v2_bit_512`